# Exp 011 - HGNetV2-B0 Soundscape Supervised

## Idea

Turn the strong `0.856` BirdCLEF 2026 reference into a repository-native branch that is cheaper and simpler than the Perch and noisy-student stacks.

## What We Want To Learn

- Whether `HGNetV2-B0` is a stronger supervised backbone than the current EfficientNet-B0 native branch.
- Whether a unified dataframe built from `train_audio + labeled soundscape clips` transfers better than our previous staged finetuning path.
- Whether optional wav-cache IO and `soundfile.SoundFile` partial reads give us a fast enough supervised branch to iterate on repeatedly.

## Success Criteria

- A stronger local metric than the current short-context supervised native path.
- Better soundscape-only validation coverage than the sparse one-fold readouts from earlier experiments.
- A clean artifact set we can later export into a dedicated Kaggle inference notebook.


## Plan

1. Load taxonomy, `train.csv`, and `train_soundscapes_labels.csv`.
2. Build explicit soundscape-clip supervision by merging contiguous same-label intervals.
3. Optionally materialize soundscape clip wav files and optionally build a `train_audio` wav cache.
4. Build a unified training dataframe across isolated audio and soundscape clips.
5. Split with grouped multi-label folds by `audio_id`.
6. Train `HGNetV2-B0` on 5-second crops with log-mel features and spectrogram MixUp.
7. Track both overall validation ROC-AUC and soundscape-only ROC-AUC.
8. Export the best checkpoint, validation metadata, and validation predictions for later OOF analysis.


In [53]:
import ast
import gc
import json
import math
import os
import random
import re
import typing as tp
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'data').exists():
            return candidate
    return start


ROOT = find_repo_root()
DATA = ROOT / 'data' / 'birdclef-2026'
PROCESSED = ROOT / 'processed_data'
NOTEBOOKS = ROOT / 'notebooks'
EXPERIMENTS = ROOT / 'experiments'


In [54]:
@dataclass
class Config:
    experiment_id: str = 'exp_011'
    experiment_name: str = 'hgnetv2_soundscape_supervised'
    fold: int = 3
    n_folds: int = 4
    random_seed: int = 1086

    sample_rate: int = 32_000
    segment_seconds: int = 5
    n_fft: int = 2048
    win_length: int = 626
    hop_length: int = 313
    f_min: int = 20
    n_mels: int = 256
    top_db: float = 80.0
    image_size: tuple[int, int] = (256, 256)

    model_name: str = 'hgnetv2_b0.ssld_stage2_ft_in1k'
    pretrained: bool = True
    drop_path_rate: float = 0.0

    epochs: int = 12
    warmup_epochs: int = 3
    batch_size: int = 24
    learning_rate: float = 5e-4
    weight_decay: float = 1e-4
    num_workers: int = 0
    use_amp: bool = True
    mixup_alpha: float = 1.0
    mixup_theta: float = 0.8

    soundscape_merge_gap_sec: int = 0
    prefer_train_audio_wav_cache: bool = True
    export_soundscape_wavs: bool = False
    build_train_audio_wav_cache: bool = False
    max_train_audio_cache_files: int | None = None

    max_rows_total: int | None = None
    max_train_rows: int | None = None
    max_valid_rows: int | None = None

    selection_metric: str = 'soundscape_macro_auc'
    save_every_epoch: bool = True


CFG = Config()
RUN_PREPARE_ARTIFACTS = True
RUN_TRAINING = True
RESUME_TRAINING = True

PROC_DIR = PROCESSED / CFG.experiment_id
SOUNDSCAPE_CLIP_DIR = PROC_DIR / 'train_soundscapes_split'
TRAIN_AUDIO_WAV_DIR = PROC_DIR / 'train_audio_wav'
OUTPUT_DIR = EXPERIMENTS / 'outputs' / f'{CFG.experiment_id}_{CFG.experiment_name}' / f'fold_{CFG.fold:02d}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)


In [55]:
def set_random_seed(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_random_seed(CFG.random_seed)


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = pick_device()
amp_enabled = bool(CFG.use_amp and device.type == 'cuda')


def autocast_context() -> tp.ContextManager[object]:
    if amp_enabled:
        return torch.amp.autocast('cuda', enabled=True)
    return nullcontext()


def save_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))


def sanitize_filename(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value)


print({
    'root': str(ROOT),
    'data_exists': DATA.exists(),
    'device': str(device),
    'amp_enabled': amp_enabled,
    'output_dir': str(OUTPUT_DIR),
    'soundscape_clip_dir': str(SOUNDSCAPE_CLIP_DIR),
    'train_audio_wav_dir': str(TRAIN_AUDIO_WAV_DIR),
})


{'root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026', 'data_exists': True, 'device': 'mps', 'amp_enabled': False, 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_011_hgnetv2_soundscape_supervised/fold_03', 'soundscape_clip_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/processed_data/exp_011/train_soundscapes_split', 'train_audio_wav_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/processed_data/exp_011/train_audio_wav'}


In [56]:
train_labels = pd.read_csv(DATA / 'train.csv')
train_soundscape_segments = pd.read_csv(DATA / 'train_soundscapes_labels.csv').drop_duplicates().reset_index(drop=True)
taxonomy = pd.read_csv(DATA / 'taxonomy.csv')

CLASSES = taxonomy['primary_label'].tolist()
N_CLASSES = len(CLASSES)
label2idx = {label: idx for idx, label in enumerate(CLASSES)}
idx2label = {idx: label for label, idx in label2idx.items()}

for col in ['start', 'end']:
    train_soundscape_segments[col] = pd.to_datetime(train_soundscape_segments[col], format='%H:%M:%S')
train_soundscape_segments['start_sec'] = train_soundscape_segments['start'].dt.minute * 60 + train_soundscape_segments['start'].dt.second
train_soundscape_segments['end_sec'] = train_soundscape_segments['end'].dt.minute * 60 + train_soundscape_segments['end'].dt.second

raw_soundscape_tokens = sorted({
    token.strip()
    for label_string in train_soundscape_segments['primary_label'].tolist()
    for token in str(label_string).split(';')
    if token.strip()
})
unknown_soundscape_labels = sorted(set(raw_soundscape_tokens) - set(CLASSES))

expanded_rows = []
for row in train_soundscape_segments.itertuples(index=False):
    labels = [token.strip() for token in str(row.primary_label).split(';') if token.strip() and token.strip() in label2idx]
    for label in labels:
        expanded_rows.append({
            'filename': row.filename,
            'start': row.start,
            'end': row.end,
            'start_sec': int(row.start_sec),
            'end_sec': int(row.end_sec),
            'primary_label': label,
        })
train_soundscape_labels = pd.DataFrame(expanded_rows)

summary_df = pd.DataFrame([
    {'table': 'train_audio', 'rows': int(len(train_labels)), 'classes': int(train_labels['primary_label'].nunique())},
    {'table': 'train_soundscape_segments_unique', 'rows': int(len(train_soundscape_segments)), 'classes': int(len(raw_soundscape_tokens))},
    {'table': 'train_soundscape_label_rows_expanded', 'rows': int(len(train_soundscape_labels)), 'classes': int(train_soundscape_labels['primary_label'].nunique()) if not train_soundscape_labels.empty else 0},
    {'table': 'taxonomy', 'rows': int(len(taxonomy)), 'classes': int(len(taxonomy))},
])

display(summary_df)
print({
    'unknown_soundscape_labels_dropped': len(unknown_soundscape_labels),
    'known_soundscape_target_labels': int(len(set(raw_soundscape_tokens) & set(CLASSES))),
})


,table,rows,classes
0,train_audio,35549,206
1,train_soundscape_segments_unique,739,75
2,train_soundscape_label_rows_expanded,3122,75
3,taxonomy,234,234


{'unknown_soundscape_labels_dropped': 0, 'known_soundscape_target_labels': 75}


In [57]:
TRAIN_AUDIO_DIR = DATA / 'train_audio'
TRAIN_SOUNDSCAPE_DIR = DATA / 'train_soundscapes'


def merge_contiguous_soundscape_intervals(label_df: pd.DataFrame) -> pd.DataFrame:
    merged_rows: list[dict[str, tp.Any]] = []
    ordered = label_df.sort_values(['filename', 'primary_label', 'start_sec', 'end_sec']).reset_index(drop=True)
    for (filename, primary_label), group in ordered.groupby(['filename', 'primary_label'], sort=False):
        current_start = None
        current_end = None
        for row in group.itertuples(index=False):
            start_sec = int(row.start_sec)
            end_sec = int(row.end_sec)
            if current_start is None:
                current_start = start_sec
                current_end = end_sec
                continue
            if start_sec <= current_end + CFG.soundscape_merge_gap_sec:
                current_end = max(current_end, end_sec)
            else:
                merged_rows.append({
                    'audio_id': Path(filename).stem,
                    'filename': filename,
                    'primary_label': primary_label,
                    'labels': primary_label,
                    'source': 'soundscape_clip',
                    'source_file_path': str(TRAIN_SOUNDSCAPE_DIR / filename),
                    'start_sec': current_start,
                    'end_sec': current_end,
                })
                current_start = start_sec
                current_end = end_sec
        if current_start is not None:
            merged_rows.append({
                'audio_id': Path(filename).stem,
                'filename': filename,
                'primary_label': primary_label,
                'labels': primary_label,
                'source': 'soundscape_clip',
                'source_file_path': str(TRAIN_SOUNDSCAPE_DIR / filename),
                'start_sec': current_start,
                'end_sec': current_end,
            })
    manifest = pd.DataFrame(merged_rows)
    if manifest.empty:
        return manifest
    manifest['clip_start_frame'] = (manifest['start_sec'] * CFG.sample_rate).astype(int)
    manifest['clip_end_frame'] = (manifest['end_sec'] * CFG.sample_rate).astype(int)
    manifest['clip_duration_sec'] = manifest['end_sec'] - manifest['start_sec']
    manifest['materialized_path'] = pd.Series([pd.NA] * len(manifest), dtype='object')
    manifest['cache_mode'] = 'offset_read'
    return manifest


def export_soundscape_clips(manifest: pd.DataFrame, clip_dir: Path) -> pd.DataFrame:
    clip_dir.mkdir(parents=True, exist_ok=True)
    updated = manifest.copy()
    for idx, row in tqdm(list(updated.iterrows()), total=len(updated), desc='export soundscape clips'):
        out_name = sanitize_filename(f"{Path(row['filename']).stem}_{row['primary_label']}_{int(row['start_sec']):04d}-{int(row['end_sec']):04d}.wav")
        out_path = clip_dir / out_name
        if not out_path.exists():
            with sf.SoundFile(row['source_file_path']) as snd:
                snd.seek(int(row['clip_start_frame']))
                wave = snd.read(frames=int(row['clip_end_frame'] - row['clip_start_frame']), dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)
            sf.write(out_path, wave, CFG.sample_rate, format='WAV', subtype='FLOAT')
        updated.at[idx, 'materialized_path'] = str(out_path)
        updated.at[idx, 'cache_mode'] = 'materialized_clip'
        updated.at[idx, 'clip_start_frame'] = 0
        updated.at[idx, 'clip_end_frame'] = int((row['end_sec'] - row['start_sec']) * CFG.sample_rate)
    return updated


soundscape_clip_manifest = merge_contiguous_soundscape_intervals(train_soundscape_labels)
if CFG.export_soundscape_wavs and RUN_PREPARE_ARTIFACTS:
    soundscape_clip_manifest = export_soundscape_clips(soundscape_clip_manifest, SOUNDSCAPE_CLIP_DIR)

soundscape_clip_manifest['file_path'] = soundscape_clip_manifest['materialized_path'].fillna(soundscape_clip_manifest['source_file_path'])
soundscape_clip_manifest.to_csv(PROC_DIR / 'soundscape_clip_manifest.csv', index=False)

clip_summary = pd.DataFrame([
    {'metric': 'merged_soundscape_clips', 'value': int(len(soundscape_clip_manifest))},
    {'metric': 'materialized_soundscape_clips', 'value': int(soundscape_clip_manifest['materialized_path'].notna().sum())},
    {'metric': 'soundscape_active_classes', 'value': int(soundscape_clip_manifest['primary_label'].nunique())},
])

display(clip_summary)
display(soundscape_clip_manifest.head())


,metric,value
0,merged_soundscape_clips,529
1,materialized_soundscape_clips,0
2,soundscape_active_classes,75


,audio_id,filename,primary_label,labels,source,source_file_path,start_sec,end_sec,clip_start_frame,clip_end_frame,clip_duration_sec,materialized_path,cache_mode,file_path
0,BC2026_Train_0001_S08_20250606_030007,BC2026_Train_0001_S08_20250606_030007.ogg,116570,116570,soundscape_clip,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...,50,60,1600000,1920000,10,<NA>,offset_read,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...
1,BC2026_Train_0001_S08_20250606_030007,BC2026_Train_0001_S08_20250606_030007.ogg,47158son05,47158son05,soundscape_clip,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...,55,60,1760000,1920000,5,<NA>,offset_read,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...
2,BC2026_Train_0001_S08_20250606_030007,BC2026_Train_0001_S08_20250606_030007.ogg,47158son10,47158son10,soundscape_clip,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...,20,60,640000,1920000,40,<NA>,offset_read,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...
3,BC2026_Train_0001_S08_20250606_030007,BC2026_Train_0001_S08_20250606_030007.ogg,47158son13,47158son13,soundscape_clip,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...,0,60,0,1920000,60,<NA>,offset_read,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...
4,BC2026_Train_0001_S08_20250606_030007,BC2026_Train_0001_S08_20250606_030007.ogg,47158son17,47158son17,soundscape_clip,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...,0,60,0,1920000,60,<NA>,offset_read,/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026...


In [58]:
def original_train_audio_path(filename: str) -> Path:
    return TRAIN_AUDIO_DIR / filename


def train_audio_cache_subdir(primary_label: str, filename: str) -> Path:
    rel = Path(filename)
    if len(rel.parts) > 1:
        return rel.parent
    return Path(primary_label)


def expected_train_audio_wav_path(primary_label: str, filename: str) -> Path:
    return TRAIN_AUDIO_WAV_DIR / train_audio_cache_subdir(primary_label, filename) / f"{Path(filename).stem}.wav"


def build_train_audio_wav_cache(label_df: pd.DataFrame, limit: int | None = None) -> pd.DataFrame:
    unique_rows = label_df[['filename', 'primary_label']].drop_duplicates().reset_index(drop=True)
    if limit is not None:
        unique_rows = unique_rows.head(limit)
    results: list[dict[str, tp.Any]] = []
    for row in tqdm(list(unique_rows.itertuples(index=False)), total=len(unique_rows), desc='build train_audio wav cache'):
        src_path = original_train_audio_path(row.filename)
        out_path = expected_train_audio_wav_path(row.primary_label, row.filename)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        if not out_path.exists():
            wave, sample_rate = sf.read(src_path, dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)
            sf.write(out_path, wave, sample_rate, format='WAV', subtype='FLOAT')
        results.append({
            'filename': row.filename,
            'primary_label': row.primary_label,
            'wav_path': str(out_path),
        })
    cache_df = pd.DataFrame(results)
    cache_df.to_csv(PROC_DIR / 'train_audio_wav_cache_manifest.csv', index=False)
    return cache_df


if CFG.build_train_audio_wav_cache and RUN_PREPARE_ARTIFACTS:
    _ = build_train_audio_wav_cache(train_labels, CFG.max_train_audio_cache_files)


def resolve_train_audio_path(primary_label: str, filename: str) -> str:
    if CFG.prefer_train_audio_wav_cache:
        wav_path = expected_train_audio_wav_path(primary_label, filename)
        if wav_path.exists():
            return str(wav_path)
    return str(original_train_audio_path(filename))


In [59]:
def build_train_audio_frame(label_df: pd.DataFrame) -> pd.DataFrame:
    frame = pd.DataFrame({
        'audio_id': label_df['filename'].map(lambda x: Path(x).stem),
        'filename': label_df['filename'],
        'primary_label': label_df['primary_label'],
        'secondary_labels': label_df['secondary_labels'],
    })
    frame['labels'] = [
        ';'.join([primary] + [label for label in ast.literal_eval(secondary) if label in label2idx])
        for primary, secondary in frame[['primary_label', 'secondary_labels']].itertuples(index=False)
    ]
    frame['source'] = 'train_audio'
    frame['file_path'] = [
        resolve_train_audio_path(primary_label, filename)
        for primary_label, filename in frame[['primary_label', 'filename']].itertuples(index=False)
    ]
    frame['source_file_path'] = [
        str(original_train_audio_path(filename))
        for filename in frame['filename'].tolist()
    ]
    frame['clip_start_frame'] = 0
    frame['clip_end_frame'] = -1
    frame['clip_duration_sec'] = np.nan
    frame['cache_mode'] = frame['file_path'].map(lambda x: 'wav_cache' if x.endswith('.wav') else 'ogg_offset')
    return frame[['audio_id', 'filename', 'primary_label', 'labels', 'source', 'file_path', 'source_file_path', 'clip_start_frame', 'clip_end_frame', 'clip_duration_sec', 'cache_mode']]


def build_multihot(labels: pd.Series, mapping: dict[str, int]) -> np.ndarray:
    arr = np.zeros((len(labels), len(mapping)), dtype=np.float32)
    for row_idx, label_string in enumerate(labels.tolist()):
        for label in str(label_string).split(';'):
            if label in mapping:
                arr[row_idx, mapping[label]] = 1.0
    return arr


train_audio_frame = build_train_audio_frame(train_labels)
train_df = pd.concat([
    train_audio_frame,
    soundscape_clip_manifest[['audio_id', 'filename', 'primary_label', 'labels', 'source', 'file_path', 'source_file_path', 'clip_start_frame', 'clip_end_frame', 'clip_duration_sec', 'cache_mode']],
], axis=0, ignore_index=True)

if CFG.max_rows_total is not None:
    train_df = train_df.sample(min(CFG.max_rows_total, len(train_df)), random_state=CFG.random_seed).reset_index(drop=True)

labels_arr = build_multihot(train_df['labels'], label2idx)
train_df.to_csv(PROC_DIR / 'train_df.csv', index=False)
np.save(PROC_DIR / 'train_labels_multihot.npy', labels_arr)

source_summary = (
    train_df.groupby('source')
    .agg(rows=('audio_id', 'size'), unique_audio=('audio_id', 'nunique'))
    .reset_index()
)

display(source_summary)
print('train_df shape:', train_df.shape)
print('labels_arr shape:', labels_arr.shape)


,source,rows,unique_audio
0,soundscape_clip,529,66
1,train_audio,35549,35549


train_df shape: (36078, 11)
labels_arr shape: (36078, 234)


In [60]:
class MultiLabelStratifiedGroupKFold:
    def __init__(self, n_splits: int, random_state: int):
        self.n_splits = n_splits
        self.random_state = random_state

    def split(self, label_arr: np.ndarray, group_ids: np.ndarray):
        np.random.seed(self.random_state)
        random.seed(self.random_state)

        unique_groups = sorted(set(group_ids))
        group_to_idx = {group_id: idx for idx, group_id in enumerate(unique_groups)}
        group_index_arr = np.vectorize(group_to_idx.get)(group_ids)

        n_groups = len(unique_groups)
        n_classes = label_arr.shape[1]
        class_totals = label_arr.sum(axis=0)
        class_totals[class_totals == 0] = 1

        counts_by_group = np.zeros((n_groups, n_classes), dtype=np.int64)
        for row_idx, group_idx in enumerate(group_index_arr):
            counts_by_group[group_idx] += label_arr[row_idx].astype(np.int64)

        counts_by_fold = np.zeros((self.n_splits, n_classes), dtype=np.int64)
        groups_by_fold: list[list[int]] = [[] for _ in range(self.n_splits)]

        group_items = list(enumerate(counts_by_group))
        random.shuffle(group_items)

        for group_idx, group_count in sorted(group_items, key=lambda item: -np.std(item[1])):
            best_fold = None
            best_score = None
            for fold_idx in range(self.n_splits):
                counts_by_fold[fold_idx] += group_count
                fold_score = (counts_by_fold / class_totals).std(axis=0).mean()
                counts_by_fold[fold_idx] -= group_count
                if best_score is None or fold_score < best_score:
                    best_score = fold_score
                    best_fold = fold_idx
            counts_by_fold[best_fold] += group_count
            groups_by_fold[best_fold].append(group_idx)

        row_indices = np.arange(label_arr.shape[0])
        for fold_idx in range(self.n_splits):
            valid_groups = groups_by_fold[fold_idx]
            valid_mask = np.isin(group_index_arr, valid_groups)
            yield row_indices[~valid_mask], row_indices[valid_mask]


splitter = MultiLabelStratifiedGroupKFold(n_splits=CFG.n_folds, random_state=CFG.random_seed)
splits = list(splitter.split(labels_arr, train_df['audio_id'].to_numpy()))
train_df = train_df.copy()
train_df['fold'] = -1
for fold_idx, (_, valid_idx) in enumerate(splits):
    train_df.loc[valid_idx, 'fold'] = fold_idx

fold_summary = (
    train_df.groupby(['fold', 'source'])
    .agg(rows=('audio_id', 'size'), unique_audio=('audio_id', 'nunique'))
    .reset_index()
    .sort_values(['fold', 'source'])
    .reset_index(drop=True)
)
fold_summary.to_csv(PROC_DIR / 'fold_summary.csv', index=False)
train_df.to_csv(PROC_DIR / 'train_df_with_folds.csv', index=False)

display(fold_summary)


,fold,source,rows,unique_audio
0,0,soundscape_clip,136,17
1,0,train_audio,8946,8946
2,1,soundscape_clip,124,15
3,1,train_audio,8911,8911
4,2,soundscape_clip,132,16
5,2,train_audio,8875,8875
6,3,soundscape_clip,137,18
7,3,train_audio,8817,8817


## Training Definitions

The implementation below keeps the strong reference ideas intact while staying notebook-native:

- `soundfile.SoundFile` partial reads instead of loading full files into memory.
- Optional wav-cache for `train_audio` and optional materialized soundscape clip wavs.
- Log-mel frontend and spectrogram MixUp.
- Grouped multi-label validation with both overall and soundscape-only metrics.
- Resume-from-checkpoint support because this branch is designed for longer reruns.


In [61]:
def build_fold_frame(frame: pd.DataFrame, label_arr: np.ndarray, fold_id: int):
    train_mask = frame['fold'] != fold_id
    valid_mask = frame['fold'] == fold_id

    train_frame = frame.loc[train_mask].reset_index(drop=True)
    valid_frame = frame.loc[valid_mask].reset_index(drop=True)
    train_labels = label_arr[train_mask.to_numpy()]
    valid_labels = label_arr[valid_mask.to_numpy()]

    if CFG.max_train_rows is not None:
        keep = min(CFG.max_train_rows, len(train_frame))
        train_frame = train_frame.sample(keep, random_state=CFG.random_seed).reset_index(drop=True)
        train_labels = build_multihot(train_frame['labels'], label2idx)
    if CFG.max_valid_rows is not None:
        keep = min(CFG.max_valid_rows, len(valid_frame))
        valid_frame = valid_frame.sample(keep, random_state=CFG.random_seed).reset_index(drop=True)
        valid_labels = build_multihot(valid_frame['labels'], label2idx)

    return train_frame, valid_frame, train_labels.astype(np.float32), valid_labels.astype(np.float32)


def read_audio_region(path: str, clip_start_frame: int, clip_end_frame: int, sample_frames: int, training: bool) -> np.ndarray:
    with sf.SoundFile(path) as snd:
        total_frames = snd.frames
        region_start = max(int(clip_start_frame), 0)
        region_end = total_frames if int(clip_end_frame) <= 0 else min(int(clip_end_frame), total_frames)
        region_frames = max(region_end - region_start, 0)

        if region_frames <= 0:
            return np.zeros(sample_frames, dtype=np.float32)

        if region_frames >= sample_frames:
            if training:
                offset = np.random.randint(region_frames - sample_frames + 1)
            else:
                offset = 0
            snd.seek(region_start + offset)
            wave = snd.read(frames=sample_frames, dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)
            if wave.shape[0] == sample_frames:
                return wave
        else:
            snd.seek(region_start)
            wave = snd.read(frames=region_frames, dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)

        actual_frames = int(wave.shape[0])
        if actual_frames >= sample_frames:
            return wave[:sample_frames]

        padded = np.zeros(sample_frames, dtype=np.float32)
        pad_start = np.random.randint(sample_frames - actual_frames + 1) if training else 0
        padded[pad_start: pad_start + actual_frames] = wave
        return padded


class UnifiedBirdDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, label_arr: np.ndarray, training: bool):
        self.frame = frame.reset_index(drop=True)
        self.labels = label_arr.astype(np.float32)
        self.training = training
        self.sample_frames = CFG.sample_rate * CFG.segment_seconds

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict[str, tp.Any]:
        row = self.frame.iloc[idx]
        wave = read_audio_region(
            path=str(row['file_path']),
            clip_start_frame=int(row['clip_start_frame']),
            clip_end_frame=int(row['clip_end_frame']),
            sample_frames=self.sample_frames,
            training=self.training,
        )
        return {
            'wave': wave,
            'label': self.labels[idx],
            'index': idx,
        }


class LogMelSpectrogramTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate,
            n_fft=CFG.n_fft,
            win_length=CFG.win_length,
            hop_length=CFG.hop_length,
            f_min=CFG.f_min,
            n_mels=CFG.n_mels,
            power=2.0,
            center=True,
            pad_mode='reflect',
            norm='slaney',
            mel_scale='htk',
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=CFG.top_db)

    @torch.no_grad()
    def forward(self, wave: torch.Tensor) -> torch.Tensor:
        if wave.ndim == 1:
            wave = wave.unsqueeze(0)
        mel = self.mel(wave)
        mel = self.db(mel)
        mel = mel.unsqueeze(1)
        mel = F.interpolate(mel, size=CFG.image_size, mode='bilinear', align_corners=False)
        flat = mel.flatten(start_dim=1)
        min_val = flat.min(dim=1).values[:, None, None, None]
        max_val = flat.max(dim=1).values[:, None, None, None]
        mel = (mel - min_val) / (max_val - min_val + 1e-7)
        return mel


class SpectrogramMixUp(nn.Module):
    def __init__(self, alpha: float = 1.0, theta: float = 0.8):
        super().__init__()
        self.beta_dist = torch.distributions.beta.Beta(alpha, alpha)
        self.theta = theta

    def forward(self, image: torch.Tensor, label: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size = image.shape[0]
        if batch_size < 2:
            return image, label
        lambdas = self.beta_dist.sample((batch_size,)).to(image.device)
        lambdas = torch.maximum(lambdas, 1 - lambdas).float()
        perm = torch.randperm(batch_size, device=image.device)
        image = lambdas[:, None, None, None] * image + (1 - lambdas[:, None, None, None]) * image[perm]
        label = lambdas[:, None] * label + (1 - lambdas[:, None]) * label[perm]
        label = torch.where(label >= self.theta, torch.ones_like(label), label)
        return image, label


In [62]:
def make_loader(dataset: Dataset, training: bool) -> DataLoader:
    kwargs = dict(
        dataset=dataset,
        batch_size=CFG.batch_size,
        shuffle=training,
        drop_last=training and len(dataset) >= CFG.batch_size,
        num_workers=CFG.num_workers,
        pin_memory=(device.type == 'cuda'),
    )
    if CFG.num_workers > 0:
        kwargs['persistent_workers'] = training
        kwargs['prefetch_factor'] = 2
    return DataLoader(**kwargs)


def compute_macro_auc(y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float | int | None]:
    positive_mask = y_true.sum(axis=0) > 0
    negative_mask = (y_true.shape[0] - y_true.sum(axis=0)) > 0
    scored_mask = positive_mask & negative_mask
    scored_classes = int(scored_mask.sum())
    skipped_no_positive = int((~positive_mask).sum())
    skipped_no_negative = int((~negative_mask).sum())
    if scored_classes == 0:
        return {
            'macro_auc': np.nan,
            'scored_classes': scored_classes,
            'skipped_no_positive': skipped_no_positive,
            'skipped_no_negative': skipped_no_negative,
        }
    macro_auc = roc_auc_score(y_true[:, scored_mask], y_score[:, scored_mask], average='macro')
    return {
        'macro_auc': float(macro_auc),
        'scored_classes': scored_classes,
        'skipped_no_positive': skipped_no_positive,
        'skipped_no_negative': skipped_no_negative,
    }


def evaluate_predictions(valid_frame: pd.DataFrame, y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float | int | None]:
    overall = compute_macro_auc(y_true, y_score)
    soundscape_mask = valid_frame['source'].eq('soundscape_clip').to_numpy()
    soundscape_metrics = {
        'soundscape_macro_auc': np.nan,
        'soundscape_scored_classes': 0,
        'soundscape_skipped_no_positive': 0,
        'soundscape_skipped_no_negative': 0,
    }
    if soundscape_mask.any():
        tmp = compute_macro_auc(y_true[soundscape_mask], y_score[soundscape_mask])
        soundscape_metrics = {
            'soundscape_macro_auc': tmp['macro_auc'],
            'soundscape_scored_classes': tmp['scored_classes'],
            'soundscape_skipped_no_positive': tmp['skipped_no_positive'],
            'soundscape_skipped_no_negative': tmp['skipped_no_negative'],
        }
    return {**overall, **soundscape_metrics}


def build_model() -> nn.Module:
    model = timm.create_model(
        CFG.model_name,
        pretrained=CFG.pretrained,
        in_chans=1,
        num_classes=N_CLASSES,
        drop_path_rate=CFG.drop_path_rate,
    )
    return model


def load_resume_payload(output_dir: Path) -> dict[str, tp.Any] | None:
    last_path = output_dir / 'last_model.pt'
    if not last_path.exists():
        return None
    return torch.load(last_path, map_location='cpu')


In [63]:
def train_one_fold(train_frame: pd.DataFrame, valid_frame: pd.DataFrame, train_labels: np.ndarray, valid_labels: np.ndarray, output_dir: Path) -> tuple[pd.DataFrame, dict[str, tp.Any]]:
    output_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = UnifiedBirdDataset(train_frame, train_labels, training=True)
    valid_dataset = UnifiedBirdDataset(valid_frame, valid_labels, training=False)
    train_loader = make_loader(train_dataset, training=True)
    valid_loader = make_loader(valid_dataset, training=False)

    mel_transform = LogMelSpectrogramTransform().to(device).eval()
    mixup = SpectrogramMixUp(alpha=CFG.mixup_alpha, theta=CFG.mixup_theta)
    model = build_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    scheduler = OneCycleLR(
        optimizer=optimizer,
        max_lr=CFG.learning_rate,
        epochs=CFG.epochs,
        steps_per_epoch=max(1, len(train_loader)),
        pct_start=max(1, CFG.warmup_epochs) / max(1, CFG.epochs),
        div_factor=25,
        final_div_factor=4.0,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    loss_fn = nn.BCEWithLogitsLoss()

    history: list[dict[str, tp.Any]] = []
    start_epoch = 1
    best_metric = -float('inf')
    resume_mode = 'fresh'

    if RESUME_TRAINING:
        payload = load_resume_payload(output_dir)
        if payload is not None:
            model.load_state_dict(payload['model_state_dict'])
            optimizer.load_state_dict(payload['optimizer_state_dict'])
            scheduler.load_state_dict(payload['scheduler_state_dict'])
            scaler_state = payload.get('scaler_state_dict')
            if scaler_state is not None and amp_enabled:
                scaler.load_state_dict(scaler_state)
            history = payload.get('history', [])
            start_epoch = int(payload.get('epoch', 0)) + 1
            best_metric = float(payload.get('best_metric', -float('inf')))
            resume_mode = 'resume_exp011'

    best_payload: dict[str, tp.Any] | None = None
    best_valid_outputs: dict[str, np.ndarray] | None = None

    for epoch in range(start_epoch, CFG.epochs + 1):
        model.train()
        train_loss_sum = 0.0
        for batch in tqdm(train_loader, desc=f'epoch {epoch} train', leave=False):
            wave = batch['wave'].to(device, dtype=torch.float32)
            label = batch['label'].to(device, dtype=torch.float32)
            optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                image = mel_transform(wave)
            if epoch > CFG.warmup_epochs:
                image, label = mixup(image, label)
            with autocast_context():
                logits = model(image)
                loss = loss_fn(logits, label)
            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            scheduler.step()
            train_loss_sum += float(loss.item())
            del wave, label, image, logits, loss
        train_loss = train_loss_sum / max(1, len(train_loader))

        model.eval()
        valid_loss_sum = 0.0
        logits_parts: list[np.ndarray] = []
        label_parts: list[np.ndarray] = []
        index_parts: list[np.ndarray] = []
        for batch in tqdm(valid_loader, desc=f'epoch {epoch} valid', leave=False):
            wave = batch['wave'].to(device, dtype=torch.float32)
            label = batch['label'].to(device, dtype=torch.float32)
            with torch.no_grad():
                image = mel_transform(wave)
                with autocast_context():
                    logits = model(image)
                    loss = loss_fn(logits, label)
            valid_loss_sum += float(loss.item())
            logits_parts.append(logits.detach().float().cpu().numpy())
            label_parts.append(label.detach().float().cpu().numpy())
            index_parts.append(batch['index'].detach().cpu().numpy())
            del wave, label, image, logits, loss
        valid_loss = valid_loss_sum / max(1, len(valid_loader))

        logits_all = np.concatenate(logits_parts, axis=0)
        labels_all = np.concatenate(label_parts, axis=0)
        index_all = np.concatenate(index_parts, axis=0)
        probs_all = 1.0 / (1.0 + np.exp(-logits_all))

        order = np.argsort(index_all)
        logits_all = logits_all[order]
        labels_all = labels_all[order]
        probs_all = probs_all[order]
        valid_metrics = evaluate_predictions(valid_frame.reset_index(drop=True), labels_all, probs_all)
        selected_metric = valid_metrics['soundscape_macro_auc']
        if np.isnan(selected_metric):
            selected_metric = valid_metrics['macro_auc']

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'macro_auc': valid_metrics['macro_auc'],
            'scored_classes': valid_metrics['scored_classes'],
            'skipped_no_positive': valid_metrics['skipped_no_positive'],
            'skipped_no_negative': valid_metrics['skipped_no_negative'],
            'soundscape_macro_auc': valid_metrics['soundscape_macro_auc'],
            'soundscape_scored_classes': valid_metrics['soundscape_scored_classes'],
            'soundscape_skipped_no_positive': valid_metrics['soundscape_skipped_no_positive'],
            'soundscape_skipped_no_negative': valid_metrics['soundscape_skipped_no_negative'],
            'valid_loss': valid_loss,
            'learning_rate': float(scheduler.get_last_lr()[0]),
            'selection_metric': float(selected_metric),
        }
        history.append(row)
        history_df = pd.DataFrame(history)
        history_df.to_csv(output_dir / 'history.csv', index=False)

        payload = {
            'epoch': epoch,
            'best_metric': max(best_metric, float(selected_metric)),
            'history': history,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if amp_enabled else None,
            'cfg': asdict(CFG),
            'classes': CLASSES,
            'resume_mode': resume_mode,
        }
        if CFG.save_every_epoch:
            torch.save(payload, output_dir / 'last_model.pt')

        if float(selected_metric) > best_metric:
            best_metric = float(selected_metric)
            best_payload = payload
            best_valid_outputs = {
                'logits': logits_all.astype(np.float32),
                'probs': probs_all.astype(np.float32),
                'labels': labels_all.astype(np.float32),
            }
            torch.save(best_payload, output_dir / 'best_model.pt')
            np.savez_compressed(output_dir / 'best_valid_outputs.npz', **best_valid_outputs)
            valid_frame.reset_index(drop=True).to_csv(output_dir / 'best_valid_meta.csv', index=False)

        print(row)

    history_df = pd.DataFrame(history)
    best_row = history_df.loc[history_df['selection_metric'].idxmax()].to_dict()
    snapshot = {
        'experiment_id': CFG.experiment_id,
        'experiment_name': CFG.experiment_name,
        'fold': CFG.fold,
        'best_epoch': int(best_row['epoch']),
        'best_selection_metric': float(best_row['selection_metric']),
        'best_macro_auc': float(best_row['macro_auc']),
        'best_soundscape_macro_auc': float(best_row['soundscape_macro_auc']) if not pd.isna(best_row['soundscape_macro_auc']) else None,
        'scored_classes': int(best_row['scored_classes']),
        'soundscape_scored_classes': int(best_row['soundscape_scored_classes']),
        'skipped_no_positive': int(best_row['skipped_no_positive']),
        'skipped_no_negative': int(best_row['skipped_no_negative']),
        'best_valid_loss': float(best_row['valid_loss']),
        'train_rows': int(len(train_frame)),
        'valid_rows': int(len(valid_frame)),
        'train_soundscape_rows': int(train_frame['source'].eq('soundscape_clip').sum()),
        'valid_soundscape_rows': int(valid_frame['source'].eq('soundscape_clip').sum()),
        'resume_mode': resume_mode,
        'output_dir': str(output_dir),
        'model_name': CFG.model_name,
        'device': str(device),
    }
    save_json(snapshot, output_dir / 'result_snapshot.json')
    return history_df, snapshot


In [64]:
train_frame, valid_frame, train_labels_fold, valid_labels_fold = build_fold_frame(train_df, labels_arr, CFG.fold)

fold_overview = {
    'fold': CFG.fold,
    'train_rows': int(len(train_frame)),
    'valid_rows': int(len(valid_frame)),
    'train_sources': train_frame['source'].value_counts().to_dict(),
    'valid_sources': valid_frame['source'].value_counts().to_dict(),
    'wav_cache_rows_train': int(train_frame['cache_mode'].eq('wav_cache').sum()),
    'wav_cache_rows_valid': int(valid_frame['cache_mode'].eq('wav_cache').sum()),
}
print(json.dumps(fold_overview, indent=2))

if RUN_TRAINING:
    history_df, snapshot = train_one_fold(train_frame, valid_frame, train_labels_fold, valid_labels_fold, OUTPUT_DIR)
else:
    history_df = None
    snapshot = None


{
  "fold": 3,
  "train_rows": 27124,
  "valid_rows": 8954,
  "train_sources": {
    "train_audio": 26732,
    "soundscape_clip": 392
  },
  "valid_sources": {
    "train_audio": 8817,
    "soundscape_clip": 137
  },
  "wav_cache_rows_train": 1,
  "wav_cache_rows_valid": 1
}


epoch 1 train:   0%|          | 0/1130 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [24, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


epoch 1 valid:   0%|          | 0/374 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [2, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


...

epoch 12 train:   0%|          | 0/1130 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [24, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


epoch 12 valid:   0%|          | 0/374 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [2, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


{'epoch': 12, 'train_loss': 0.015773067496453238, 'macro_auc': 0.9662342603082728, 'scored_classes': 208, 'skipped_no_positive': 26, 'skipped_no_negative': 0, 'soundscape_macro_auc': 0.7976139858742047, 'soundscape_scored_classes': 39, 'soundscape_skipped_no_positive': 195, 'soundscape_skipped_no_negative': 0, 'valid_loss': 0.010992484843298634, 'learning_rate': 5.00001180872594e-06, 'selection_metric': 0.7976139858742047}


## Results And Observations

Use this section after each run to summarize:

- best overall validation macro ROC-AUC
- best soundscape-only validation macro ROC-AUC
- whether wav-cache materially changed runtime
- whether the unified supervised branch looks stronger than the current native public baseline path


In [65]:
if (OUTPUT_DIR / 'result_snapshot.json').exists():
    snapshot = json.loads((OUTPUT_DIR / 'result_snapshot.json').read_text())
    print('Snapshot:')
    print(json.dumps(snapshot, indent=2))
    if (OUTPUT_DIR / 'history.csv').exists():
        display(pd.read_csv(OUTPUT_DIR / 'history.csv'))
else:
    print('No training artifacts yet. Set RUN_TRAINING = True to start the experiment.')


Snapshot:
{
  "experiment_id": "exp_011",
  "experiment_name": "hgnetv2_soundscape_supervised",
  "fold": 3,
  "best_epoch": 9,
  "best_selection_metric": 0.799206367013475,
  "best_macro_auc": 0.9662019378652492,
  "best_soundscape_macro_auc": 0.799206367013475,
  "scored_classes": 208,
  "soundscape_scored_classes": 39,
  "skipped_no_positive": 26,
  "skipped_no_negative": 0,
  "best_valid_loss": 0.011513754402357029,
  "train_rows": 27124,
  "valid_rows": 8954,
  "train_soundscape_rows": 392,
  "valid_soundscape_rows": 137,
  "resume_mode": "fresh",
  "output_dir": "/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_011_hgnetv2_soundscape_supervised/fold_03",
  "model_name": "hgnetv2_b0.ssld_stage2_ft_in1k",
  "device": "mps"
}


,epoch,train_loss,macro_auc,scored_classes,skipped_no_positive,skipped_no_negative,soundscape_macro_auc,soundscape_scored_classes,soundscape_skipped_no_positive,soundscape_skipped_no_negative,valid_loss,learning_rate,selection_metric
0,1,0.062907,0.660015,208,26,0,0.433479,39,195,0,0.029353,0.000140,0.433479
1,2,0.024706,0.907363,208,26,0,0.651446,39,195,0,0.019778,0.000380,0.651446
2,3,0.018808,0.934081,208,26,0,0.727561,39,195,0,0.017025,0.000500,0.727561
3,4,0.021739,0.938168,208,26,0,0.723351,39,195,0,0.015347,0.000485,0.723351
4,5,0.020646,0.951588,208,26,0,0.755712,39,195,0,0.014355,0.000442,0.755712
5,6,0.019676,0.948353,208,26,0,0.729956,39,195,0,0.013528,0.000376,0.729956
6,7,0.018728,0.956683,208,26,0,0.749557,39,195,0,0.012762,0.000295,0.749557
7,8,0.017890,0.962258,208,26,0,0.778255,39,195,0,0.011992,0.000209,0.778255
8,9,0.017131,0.966202,208,26,0,0.799206,39,195,0,0.011514,0.000129,0.799206
9,10,0.016477,0.965406,208,26,0,0.795373,39,195,0,0.011177,0.000063,0.795373
